# Precision × Batch Frontier for GPU Inference

### Throughput is a constrained optimization problem, not a dtype preference

**Question.** For a fixed model and device, which `(batch size, numerical precision)` operating points maximize throughput while satisfying memory and numerical-fidelity constraints?

The experiment treats throughput, latency, memory, and fidelity as separate measurements. It does not assume that reduced precision is safe or that the largest tested batch is optimal.


## 1 — Measurement contract

A candidate configuration is admissible only if all required constraints hold:

```text
throughput is measured after warmup
GPU work is synchronized before timing stops
peak allocated memory stays under budget
same input + same weights are used for fidelity comparison
fidelity metric is computed in a common high-precision dtype
```

The final selection is therefore a **feasible frontier problem**, not simply `argmax(throughput)`.


In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "CUDA device required to rerun this benchmark"
device="cuda"
D_MODEL=256; SEQ_LEN=64; N_LAYERS=2; VOCAB=16000; WARMUP=10; ITERS=100

class SmallTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(VOCAB,D_MODEL)
        layer=nn.TransformerEncoderLayer(D_MODEL,4,4*D_MODEL,dropout=0.0,batch_first=True,norm_first=True)
        self.enc=nn.TransformerEncoder(layer,N_LAYERS)
        self.head=nn.Linear(D_MODEL,VOCAB)
    def forward(self,x):
        return self.head(self.enc(self.embed(x)))


## 2 — Batch sweep: the tested range did **not** prove a knee

Recorded fp32 sweep:

| batch | samples/s | peak MiB | batch latency |
|---:|---:|---:|---:|
| 1 | 3,271.6 | 55 | 0.31 ms |
| 4 | 11,778.5 | 79 | 0.34 ms |
| 16 | 34,467.6 | 173 | 0.46 ms |
| 64 | 44,494.9 | 551 | 1.44 ms |

Throughput improved **13.6×** from batch 1 to 64.

Calling batch 64 the saturation ‘knee’ would be too strong. Throughput was still increasing at the largest tested point, so the evidence only establishes that the knee lies at or beyond the measured range, or requires denser samples to locate.


In [ ]:
from pathlib import Path
import json
recorded=json.loads(Path("../data/precision_batch_recorded_run.json").read_text())
rows=recorded["batch_sweep"]
print(f"throughput gain batch 1→64: {rows[-1]['throughput']/rows[0]['throughput']:.2f}x")
for prev,cur in zip(rows[:-1],rows[1:]):
    print(f"{prev['batch']:>2}→{cur['batch']:<2}: throughput {(cur['throughput']/prev['throughput']-1)*100:6.1f}% | latency {(cur['latency_ms']/prev['latency_ms']-1)*100:6.1f}%")


## 3 — Benchmark harness

The harness measures synchronized device time and peak memory. Repeated trials should be added for publication-quality confidence intervals; a single aggregate run is useful for exploration but not enough for a hardware claim.


In [ ]:
def bench(model,batch,dtype,trials=5):
    model=model.to(device=device,dtype=dtype).eval()
    x=torch.randint(0,VOCAB,(batch,SEQ_LEN),device=device)
    trial=[]
    for _ in range(trials):
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            for _ in range(WARMUP): model(x)
            torch.cuda.synchronize(); t0=time.perf_counter()
            for _ in range(ITERS): model(x)
            torch.cuda.synchronize(); dt=time.perf_counter()-t0
        trial.append({"throughput": ITERS*batch/dt, "latency_ms": 1e3*dt/ITERS, "peak_vram_mib": torch.cuda.max_memory_allocated()/2**20})
    return trial


## 4 — Precision sweep: speed is not enough

| precision | samples/s | speed vs fp32 | reported cosine vs fp32 |
|---|---:|---:|---:|
| fp32 | 8,856 | 1.00× | 1.0000 |
| fp16 | 16,653 | 1.88× | 1.0000 |
| bf16 | 20,028 | 2.26× | 1.0000 |

The throughput result is useful. The `1.0000` fidelity printout is not sufficient evidence of equivalence because four decimal places can hide meaningful error.


In [ ]:
def fidelity(reference,candidate):
    a=reference.float().flatten(); b=candidate.float().flatten(); diff=b-a
    denom=torch.linalg.vector_norm(a).clamp_min(1e-12)
    return {"cosine": F.cosine_similarity(a,b,dim=0).item(), "relative_l2": (torch.linalg.vector_norm(diff)/denom).item(), "mean_abs": diff.abs().mean().item(), "max_abs": diff.abs().max().item()}


## 5 — Same weights, same input, different arithmetic

A valid precision comparison must not instantiate independently randomized models. The fp32 parameters are the authority; candidate models receive casts of the same state dict.


In [ ]:
torch.manual_seed(42)
base=SmallTransformer().to(device).eval()
fixed_input=torch.randint(0,VOCAB,(16,SEQ_LEN),device=device)
with torch.no_grad(): reference=base(fixed_input).float()
def evaluate_dtype(dtype):
    candidate=SmallTransformer().to(device=device,dtype=dtype).eval()
    candidate.load_state_dict({k:v.to(dtype) for k,v in base.state_dict().items()})
    with torch.no_grad(): out=candidate(fixed_input)
    return fidelity(reference,out)


## 6 — Selection contract

A production choice should be selected only from configurations satisfying explicit memory, numerical-error, and latency constraints. Then choose the highest-throughput feasible point—or retain the Pareto frontier when consumers have different SLOs.


In [ ]:
def choose_config(rows, *, vram_budget_mib, max_relative_l2, max_latency_ms=None):
    feasible=[]
    for r in rows:
        if r['peak_vram_mib']>vram_budget_mib: continue
        if r.get('relative_l2',0.0)>max_relative_l2: continue
        if max_latency_ms is not None and r['latency_ms']>max_latency_ms: continue
        feasible.append(r)
    if not feasible: raise RuntimeError('no configuration satisfies the contract')
    return max(feasible,key=lambda r:r['throughput'])


## 7 — Recorded combined result

```text
precision       bf16
batch           32
throughput      29,140 samples/s
peak memory     209 MiB
memory budget   6,963 MiB
reported gain   22.54× vs fp32 @ batch 1
```

That result mixes two independent gains—batch amortization and reduced precision—so it should not be described as a precision speedup. The decomposition matters.


## 8 — What I would require before calling the configuration production-optimal

1. denser batch sweep around the saturation region;
2. repeated trials with variance / confidence intervals;
3. application-level quality metric, not logits-only cosine;
4. p50/p95/p99 request latency under realistic arrival processes;
5. memory fragmentation and long-run stability;
6. exact deployment model, sequence-length distribution, and software stack.

The transferable lesson is the method for finding a constrained operating point on a specific system.
